# 🩺 Práctica: clasificación de lesiones cutáneas con Deep Learning (Keras)

**Duración estimada:** 4 horas · **Entorno:** Google Colab con GPU (T4) · **Framework:** TensorFlow/Keras

Vamos a entrenar una red neuronal que distingue lesiones cutáneas **benignas** de **malignas** a partir de imágenes dermatoscópicas.

| Bloque | Contenido | Tiempo orientativo |
|---|---|---|
| 0 | Introducción clínica y ética | 30 min |
| 1 | Preparación del entorno y carga de datos | 20 min |
| 2 | Exploración de datos | 30 min |
| 3 | División de datos (¡cuidado con la fuga de datos!) | 30 min |
| 4 | Modelo con *transfer learning* y entrenamiento | 40 min |
| 5 | Evaluación clínica: sensibilidad, especificidad, ROC | 40 min |
| 6 | Interpretabilidad con Grad-CAM | 30 min |
| 7 | Discusión crítica | 20 min |

> ⚠️ **Aviso:** esto es una práctica docente. Ningún resultado de este notebook tiene validez clínica.

**Antes de empezar:** *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4*.

## 0. Introducción clínica y ética

Reflexiona antes de programar (puedes responder en una celda de texto):

1. En un cribado de melanoma, ¿qué es peor: un **falso negativo** (lesión maligna clasificada como benigna) o un **falso positivo**? ¿Por qué?
2. ¿Qué métrica te importa más: *accuracy*, **sensibilidad** o **especificidad**?
3. ¿Quién es responsable si el algoritmo se equivoca: el dermatólogo, el hospital, el fabricante?
4. Un dataset ¿representa a todos los pacientes? ¿Qué tonos de piel, edades y localizaciones esperas que estén infrarrepresentados?

## 1. Preparación del entorno

In [ ]:
import os, random, time, zipfile, glob
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.metrics import confusion_matrix, roc_curve, roc_auc_score, ConfusionMatrixDisplay

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print("GPU disponible:", gpus if gpus else "❌ NO (activa la GPU en el entorno de ejecución)")

### Configuración de la práctica

Hay dos fuentes de datos. Cambia `DATA_SOURCE`:

- `"ham10000"` → dataset completo (recomendado). El profesor comparte un ZIP en Drive con el CSV de metadatos y las imágenes.
- `"medmnist"` → **plan B**: DermaMNIST, se descarga con una línea y no requiere Drive.

In [ ]:
DATA_SOURCE = "ham10000"      # "ham10000" o "medmnist"
IMG_SIZE    = 224             # lado de la imagen (224 recomendado para MobileNetV2)
BATCH_SIZE  = 32
EPOCHS_HEAD = 5               # épocas entrenando solo la cabeza
EPOCHS_FT   = 3               # épocas de fine-tuning (pon 0 para saltarlo)

# ⚠️ EJERCICIO (bloque 3): pon True para dividir al azar IGNORANDO las lesiones repetidas.
# Vuelve a ejecutar desde el bloque 3 y compara los resultados.
WRONG_SPLIT = False

# Ruta del ZIP en Drive (solo para ham10000). El ZIP debe contener HAM10000_metadata.csv y las imágenes .jpg
ZIP_PATH = "/content/drive/MyDrive/practica_piel/ham10000.zip"

CLASS_NAMES = ["Benigna", "Maligna"]

### Carga de datos

**HAM10000** tiene 7 tipos de lesión (`dx`). Los agrupamos en dos clases:

- **Maligna (1):** `mel` (melanoma), `bcc` (carcinoma basocelular), `akiec` (queratosis actínica / Bowen; lesión premaligna o *in situ*, la tratamos como "a vigilar").
- **Benigna (0):** `nv` (nevus melanocítico), `bkl` (queratosis benigna), `df` (dermatofibroma), `vasc` (lesión vascular).

In [ ]:
MALIGNANT = ["mel", "bcc", "akiec"]

def load_images(paths, size=IMG_SIZE):
    def _load(p):
        im = Image.open(p)
        im.draft("RGB", (size * 2, size * 2))      # acelera la lectura de JPEG grandes
        im = im.convert("RGB").resize((size, size))
        return np.asarray(im, dtype=np.uint8)
    with ThreadPoolExecutor(8) as ex:
        return np.stack(list(ex.map(_load, paths)))

if DATA_SOURCE == "ham10000":
    from google.colab import drive
    drive.mount("/content/drive")

    LOCAL = "/content/ham10000"
    if not os.path.exists(LOCAL):
        with zipfile.ZipFile(ZIP_PATH) as z:      # copiar a disco local es MUCHO más rápido que leer de Drive
            z.extractall(LOCAL)

    meta = pd.read_csv(glob.glob(f"{LOCAL}/**/HAM10000_metadata.csv", recursive=True)[0])
    paths = {Path(p).stem: p for p in glob.glob(f"{LOCAL}/**/*.jpg", recursive=True)}
    meta["path"] = meta["image_id"].map(paths)
    meta = meta.dropna(subset=["path"]).reset_index(drop=True)
    meta["label"] = meta["dx"].isin(MALIGNANT).astype(int)
    print(f"Imágenes encontradas: {len(meta)} · lesiones únicas: {meta['lesion_id'].nunique()}")

## 2. Exploración de datos

In [ ]:
if DATA_SOURCE == "ham10000":
    print(meta["dx"].value_counts(), "\n")
    print(meta["label"].value_counts().rename({0: "Benigna", 1: "Maligna"}))
    print(f"\n% malignas: {meta['label'].mean()*100:.1f}%")

    fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
    meta["dx"].value_counts().plot.bar(ax=ax[0], title="Tipos de lesión (dx)")
    meta["label"].value_counts().rename({0: "Benigna", 1: "Maligna"}).plot.bar(ax=ax[1], title="Clase binaria")
    plt.tight_layout(); plt.show()

    # Metadatos clínicos: ¿qué sesgos ves?
    print(meta[["age", "sex", "localization"]].describe(include="all"))

In [ ]:
if DATA_SOURCE == "ham10000":
    fig, axes = plt.subplots(2, 6, figsize=(16, 6))
    for row, lab in enumerate([0, 1]):
        sample = meta[meta["label"] == lab].sample(6, random_state=SEED)
        for ax, (_, r) in zip(axes[row], sample.iterrows()):
            ax.imshow(Image.open(r["path"])); ax.set_title(f"{CLASS_NAMES[lab]} ({r['dx']})"); ax.axis("off")
    plt.tight_layout(); plt.show()

**📝 Preguntas:**
- ¿Qué proporción de lesiones es maligna? Si un modelo dijera siempre "benigna", ¿qué *accuracy* tendría?
- ¿Eres capaz de distinguir a ojo las dos clases? ¿Qué rasgos usarías (regla ABCDE)?
- ¿Qué edad y localización predominan? ¿Qué pacientes faltan?

## 3. División en entrenamiento / validación / test

> 🔑 **Punto clave:** en HAM10000 **una misma lesión puede aparecer en varias fotos** (`lesion_id`). Si repartimos las *imágenes* al azar, la misma lesión puede estar en train y en test, y el modelo "hace trampa" memorizándola. Hay que dividir **por lesión (`lesion_id`)**.

In [ ]:
if DATA_SOURCE == "ham10000":
    if WRONG_SPLIT:
        print("⚠️ División INCORRECTA (al azar por imagen)")
        train_df, tmp = train_test_split(meta, test_size=0.30, random_state=SEED, stratify=meta["label"])
        val_df, test_df = train_test_split(tmp, test_size=0.5, random_state=SEED, stratify=tmp["label"])
    else:
        print("✅ División CORRECTA (por lesion_id)")
        gss = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=SEED)
        tr_idx, tmp_idx = next(gss.split(meta, groups=meta["lesion_id"]))
        train_df, tmp = meta.iloc[tr_idx], meta.iloc[tmp_idx]
        gss2 = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=SEED)
        v_idx, te_idx = next(gss2.split(tmp, groups=tmp["lesion_id"]))
        val_df, test_df = tmp.iloc[v_idx], tmp.iloc[te_idx]

    # Comprobación de fuga de datos
    leak = set(train_df["lesion_id"]) & set(test_df["lesion_id"])
    print(f"Lesiones presentes a la vez en train y test: {len(leak)}")
    print(f"Train: {len(train_df)} · Val: {len(val_df)} · Test: {len(test_df)}")
    print("% malignas →", [round(d['label'].mean()*100, 1) for d in (train_df, val_df, test_df)])

    t0 = time.time()
    X_train, y_train = load_images(train_df["path"]), train_df["label"].values
    X_val,   y_val   = load_images(val_df["path"]),   val_df["label"].values
    X_test,  y_test  = load_images(test_df["path"]),  test_df["label"].values
    print(f"Imágenes cargadas en memoria en {time.time()-t0:.0f} s")

In [ ]:
if DATA_SOURCE == "medmnist":
    %pip install -q medmnist
    from medmnist import DermaMNIST
    def _get(split):
        ds = DermaMNIST(split=split, download=True, size=IMG_SIZE if IMG_SIZE in (28, 64, 128, 224) else 224)
        X, y = ds.imgs, ds.labels.ravel()
        # DermaMNIST: 0 akiec, 1 bcc, 2 bkl, 3 df, 4 mel, 5 nv, 6 vasc
        return X, np.isin(y, [0, 1, 4]).astype(int)
    (X_train, y_train), (X_val, y_val), (X_test, y_test) = _get("train"), _get("val"), _get("test")
    IMG_SIZE = X_train.shape[1]
    print("Train/Val/Test:", len(X_train), len(X_val), len(X_test), "· tamaño:", IMG_SIZE)
    print("⚠️ Nota: esta partición oficial NO está agrupada por lesión; los resultados pueden ser optimistas.")

In [ ]:
# Dataset de tf.data y pesos de clase (para compensar el desbalance)
def make_ds(X, y, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((X, y.astype("float32")))
    if shuffle: ds = ds.shuffle(2000, seed=SEED)
    ds = ds.batch(BATCH_SIZE).map(lambda a, b: (tf.cast(a, tf.float32), b), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)

train_ds, val_ds, test_ds = make_ds(X_train, y_train, True), make_ds(X_val, y_val), make_ds(X_test, y_test)

n = len(y_train); n_pos = y_train.sum(); n_neg = n - n_pos
class_weight = {0: n / (2 * n_neg), 1: n / (2 * n_pos)}
print("Pesos de clase:", {k: round(v, 2) for k, v in class_weight.items()})

## 4. Modelo con *transfer learning*

En lugar de entrenar una red desde cero (días de cómputo), reutilizamos **MobileNetV2**, ya entrenada con ImageNet (1,4 M de imágenes), y solo entrenamos una pequeña "cabeza" que decide benigna/maligna. Después podemos hacer *fine-tuning* de las últimas capas.

El modelo incluye *data augmentation* (giros, rotaciones, zoom, contraste), que solo se aplica durante el entrenamiento.

In [ ]:
keras.backend.clear_session()

augmentation = keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
], name="augmentation")

base = keras.applications.MobileNetV2(input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights="imagenet")
base.trainable = False

gap, drop, head = layers.GlobalAveragePooling2D(), layers.Dropout(0.3), layers.Dense(1, activation="sigmoid")

inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = augmentation(inputs)
x = layers.Rescaling(1/127.5, offset=-1)(x)       # preprocesado de MobileNetV2: [0,255] -> [-1,1]
x = base(x, training=False)
x = gap(x); x = drop(x)
outputs = head(x)
model = keras.Model(inputs, outputs)

def compile_model(lr):
    model.compile(optimizer=keras.optimizers.Adam(lr), loss="binary_crossentropy",
                  metrics=["accuracy", keras.metrics.AUC(name="auc"), keras.metrics.Recall(name="sensibilidad")])

compile_model(1e-3)
model.summary()

### Entrenamiento de la cabeza (base congelada)

Observa el tiempo por época: con GPU debería ser de segundos a pocos minutos.

In [ ]:
ckpt = keras.callbacks.ModelCheckpoint("mejor_modelo.keras", monitor="val_auc", mode="max", save_best_only=True)
t0 = time.time()
hist1 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_HEAD, class_weight=class_weight, callbacks=[ckpt])
print(f"Tiempo total: {(time.time()-t0)/60:.1f} min")

### (Opcional) *Fine-tuning* de las últimas capas

Descongelamos las últimas 30 capas de MobileNetV2 y entrenamos con una tasa de aprendizaje muy baja.

In [ ]:
hist2 = None
if EPOCHS_FT > 0:
    base.trainable = True
    for layer in base.layers[:-30]:
        layer.trainable = False
    compile_model(1e-5)                              # hay que recompilar tras cambiar 'trainable'
    t0 = time.time()
    hist2 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_FT, class_weight=class_weight, callbacks=[ckpt])
    print(f"Tiempo fine-tuning: {(time.time()-t0)/60:.1f} min")

In [ ]:
# Curvas de aprendizaje
def joined(key):
    h = hist1.history[key] + (hist2.history[key] if hist2 else [])
    return h
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
for a, (k, t) in zip(ax, [("loss", "Pérdida"), ("auc", "AUC")]):
    a.plot(joined(k), label="train"); a.plot(joined("val_" + k), label="validación")
    a.axvline(EPOCHS_HEAD - 0.5, ls="--", c="gray"); a.set_title(t); a.set_xlabel("época"); a.legend()
plt.tight_layout(); plt.show()

**📝 Preguntas:** ¿hay sobreajuste (train mejora y validación no)? ¿Qué cambia al empezar el fine-tuning (línea discontinua)?

## 5. Evaluación clínica

Evaluamos **una sola vez** sobre el conjunto de **test**, que el modelo no ha visto. Más importante que el *accuracy*:

- **Sensibilidad** = VP / (VP + FN) → de las lesiones malignas, ¿cuántas detecta?
- **Especificidad** = VN / (VN + FP) → de las benignas, ¿cuántas descarta correctamente?

In [ ]:
model = keras.models.load_model("mejor_modelo.keras")       # mejor modelo según AUC de validación
y_prob = model.predict(test_ds, verbose=0).ravel()

def report(threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    sens, spec = tp / (tp + fn), tn / (tn + fp)
    ppv, npv = tp / max(tp + fp, 1), tn / max(tn + fn, 1)
    return dict(umbral=threshold, sensibilidad=sens, especificidad=spec, VPP=ppv, VPN=npv, FN=fn, FP=fp)

r = report(0.5)
print(f"Sensibilidad: {r['sensibilidad']:.2%} · Especificidad: {r['especificidad']:.2%}")
print(f"VPP: {r['VPP']:.2%} · VPN: {r['VPN']:.2%} · Falsos negativos: {r['FN']} · Falsos positivos: {r['FP']}")
print(f"AUC: {roc_auc_score(y_test, y_prob):.3f}")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ConfusionMatrixDisplay.from_predictions(y_test, (y_prob >= 0.5).astype(int), display_labels=CLASS_NAMES, ax=ax[0], cmap="Blues")
ax[0].set_title("Matriz de confusión (umbral 0,5)")
fpr, tpr, _ = roc_curve(y_test, y_prob)
ax[1].plot(fpr, tpr, label=f"AUC = {roc_auc_score(y_test, y_prob):.3f}"); ax[1].plot([0, 1], [0, 1], "--", c="gray")
ax[1].set_xlabel("1 - Especificidad"); ax[1].set_ylabel("Sensibilidad"); ax[1].set_title("Curva ROC"); ax[1].legend()
plt.tight_layout(); plt.show()

### Elegir el umbral de decisión

El umbral 0,5 no tiene por qué ser el adecuado clínicamente. En un cribado queremos **alta sensibilidad** (que no se escape ningún melanoma), a costa de más falsos positivos.

In [ ]:
tabla = pd.DataFrame([report(t) for t in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]]).set_index("umbral")
display(tabla.style.format({"sensibilidad": "{:.1%}", "especificidad": "{:.1%}", "VPP": "{:.1%}", "VPN": "{:.1%}"}))

**📝 Preguntas:**
1. ¿Qué umbral elegirías para un cribado en Atención Primaria? ¿Y para decidir una biopsia? Justifícalo.
2. Si la prevalencia real de melanoma en consulta es mucho menor que en este dataset, ¿qué le ocurre al **VPP**?
3. *(Si hiciste el ejercicio `WRONG_SPLIT=True`)* ¿Cuánto se infló el resultado y por qué?

## 6. Interpretabilidad: Grad-CAM

Grad-CAM resalta las zonas de la imagen que más influyen en la predicción. Sirve para detectar si el modelo "mira" la lesión... o algo irrelevante (reglas, bordes, marcadores quirúrgicos, viñeteado).

In [ ]:
# Recuperamos las capas del modelo cargado
base_l = model.get_layer(base.name) if base.name in [l.name for l in model.layers] else [l for l in model.layers if isinstance(l, keras.Model) and l.name.startswith("mobilenet")][0]
gap_l  = [l for l in model.layers if isinstance(l, layers.GlobalAveragePooling2D)][0]
head_l = model.layers[-1]

def grad_cam(img_uint8):
    x = tf.convert_to_tensor(img_uint8[None].astype("float32") / 127.5 - 1.0)
    with tf.GradientTape() as tape:
        feat = base_l(x, training=False)          # mapa de características (7x7x1280)
        tape.watch(feat)
        score = head_l(gap_l(feat))               # probabilidad de "maligna"
    grads = tape.gradient(score, feat)
    weights = tf.reduce_mean(grads, axis=(1, 2), keepdims=True)
    cam = tf.nn.relu(tf.reduce_sum(weights * feat, axis=-1))[0].numpy()
    cam = cam / (cam.max() + 1e-8)
    return cam, float(score.numpy()[0, 0])

def show_gradcam(indices):
    fig, axes = plt.subplots(2, len(indices), figsize=(3.2 * len(indices), 6.5))
    for j, i in enumerate(indices):
        cam, p = grad_cam(X_test[i])
        heat = np.array(Image.fromarray((cam * 255).astype("uint8")).resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)) / 255.0
        overlay = 0.55 * X_test[i] / 255.0 + 0.45 * plt.cm.jet(heat)[..., :3]
        axes[0, j].imshow(X_test[i]); axes[0, j].set_title(f"Real: {CLASS_NAMES[y_test[i]]}\nP(maligna)={p:.2f}", fontsize=9)
        axes[1, j].imshow(overlay)
        for a in axes[:, j]: a.axis("off")
    plt.tight_layout(); plt.show()

rng = np.random.default_rng(SEED)
malig = rng.choice(np.where(y_test == 1)[0], 4, replace=False)
benig = rng.choice(np.where(y_test == 0)[0], 4, replace=False)
print("Lesiones MALIGNAS"); show_gradcam(malig)
print("Lesiones BENIGNAS"); show_gradcam(benig)

In [ ]:
# Analiza los ERRORES: falsos negativos (malignas que el modelo dejó pasar)
y_pred = (y_prob >= 0.5).astype(int)
fn_idx = np.where((y_test == 1) & (y_pred == 0))[0]
print(f"{len(fn_idx)} falsos negativos")
if len(fn_idx): show_gradcam(fn_idx[:4])

**📝 Preguntas:** ¿el modelo mira la lesión o el fondo? ¿Ves algún patrón en los falsos negativos? ¿Te fiarías del modelo si el mapa de calor señalara una esquina de la foto?

## 7. Discusión crítica

Responde brevemente en grupo:

1. **Validez externa:** este modelo se ha evaluado con imágenes del mismo dataset. ¿Funcionaría con fotos de móvil, otro dermatoscopio u otro hospital?
2. **Sesgos:** HAM10000 procede mayoritariamente de población de piel clara (Austria y Australia). ¿Qué riesgo supone para pacientes con fototipos oscuros?
3. **Etiquetas:** ¿todas las etiquetas están confirmadas por histopatología? ¿Qué pasa si hay errores en las etiquetas?
4. **Uso clínico:** ¿cómo integrarías esta herramienta en el flujo de trabajo (apoyo a la decisión, segunda opinión, cribado)? ¿Qué regulación aplica (marcado CE, MDR)?
5. **Transparencia:** ¿qué información debería acompañar a un modelo así (tarjeta del modelo, población de entrenamiento, rendimiento por subgrupos)?

## Material extra (si sobra tiempo)

- Prueba **EfficientNetB0** en lugar de MobileNetV2 (cambia `preprocess` según la red).
- Evalúa el rendimiento **por subgrupos** (sexo, edad, localización) usando `test_df`.
- Compara `WRONG_SPLIT=False` frente a `True` y cuantifica la inflación del AUC.
- Entrena con las **7 clases originales** (`softmax` + `sparse_categorical_crossentropy`).